# 🔍 Power BI — Extrator de Dados Brutos

Este notebook intercepta as requisições internas do Power BI e extrai os dados originais da base que alimenta o dashboard.

---

## 📋 Como capturar as informações necessárias (faça isso ANTES de rodar)

### Passo 1 — Abrir o DevTools no dashboard
1. Abra o dashboard público no **Chrome ou Edge**
2. Pressione **`F12`** para abrir o DevTools
3. Vá para a aba **`Network`** (Rede)
4. **Recarregue a página** com o DevTools aberto (`F5`)

### Passo 2 — Capturar o endpoint e o payload
5. No campo de filtro da aba Network, digite: **`querydata`**
6. Aguarde aparecer uma ou mais requisições na lista
7. Clique em qualquer uma delas
8. Na aba **`Headers`**: copie a **URL completa** do request → será o `API_ENDPOINT`
9. Na aba **`Payload`** → clique em **`view source`**: copie o JSON inteiro → será o `PAYLOAD_CAPTURADO`

```
Exemplo de endpoint:
https://wabi-brazil-south-api.analysis.windows.net/public/reports/querydata
                   ↑
              varia por região
```

### Passo 3 — Capturar o ResourceKey
10. O ResourceKey está na **URL pública** do dashboard:
```
https://app.powerbi.com/view?r=XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
                                ↑
                          ResourceKey está aqui
```

---

## 🗺️ Fluxo do notebook

| Célula | O que faz |
|---|---|
| **1. Instalação** | Instala dependências |
| **2. Configuração** | ⚠️ Preencha com seus dados |
| **3. Funções** | Carrega todo o código (só rodar) |
| **4. Análise do Payload** | Inspeciona o payload capturado do DevTools |
| **5. Inspeção da API** | Testa a requisição e mostra a estrutura da resposta |
| **6. Extração Completa** | Extrai todos os dados com paginação |
| **7. Visualização** | Mostra preview dos dados em DataFrame |
| **8. Download** | Baixa o CSV direto no Google Colab |

---
## Célula 1 — Instalação de dependências

In [ ]:
!pip install requests pandas --quiet
print('✅ Dependências instaladas.')

---
## Célula 2 — ⚠️ Configuração (preencha aqui)

In [24]:
# ══════════════════════════════════════════════════════════════
#  PREENCHA AS VARIÁVEIS ABAIXO COM OS DADOS DO SEU DASHBOARD
# ══════════════════════════════════════════════════════════════

# URL pública do dashboard (app.powerbi.com/view?r=...)
DASHBOARD_URL = "https://app.powerbi.com/view?r=eyJrIjoiNGExMzk1MmMtOWIyOS00N2NiLTk2MDEtZTliNWEzZWU5MDM5IiwidCI6IjNhNzhiMGNkLTdjOGUtNDkyOS04M2Q1LTE5MGE2Y2MwMTM2NSJ9"
# Endpoint capturado no DevTools (URL do POST para querydata)
API_ENDPOINT = "https://wabi-brazil-south-b-primary-api.analysis.windows.net/public/reports/querydata?synchronous=true"

# Nome da tabela base (encontrado no HTML ou na análise do payload)
TABELA_ALVO = "CONSOLIDADO_TOTAL"

# Cole aqui o JSON bruto capturado do DevTools (aba Payload → view source)
# Mantenha como None se quiser usar o payload genérico automático
PAYLOAD_CAPTURADO = {"version":"1.0.0","queries":[{"Query":{"Commands":[{"SemanticQueryDataShapeCommand":{"Query":{"Version":2,"From":[{"Name":"c","Entity":"CONSOLIDADO_TOTAL","Type":0}],"Select":[{"Aggregation":{"Expression":{"Column":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"Total do Município"}},"Function":4},"Name":"Sum(CONSOLIDADO_TOTAL.Total do Município*)","NativeReferenceName":"Máximo de Total do Município*"}],"Where":[{"Condition":{"In":{"Expressions":[{"Column":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"data extração"}}],"Values":[[{"Literal":{"Value":"datetime'2026-06-02T00:00:00'"}}]]}}}],"OrderBy":[{"Direction":2,"Expression":{"Aggregation":{"Expression":{"Column":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"Total do Município"}},"Function":4}}}]},"Binding":{"Primary":{"Groupings":[{"Projections":[0]}]},"DataReduction":{"DataVolume":3,"Primary":{"Window":{}}},"Version":1},"ExecutionMetricsKind":1}}]},"CacheKey":"{\"Commands\":[{\"SemanticQueryDataShapeCommand\":{\"Query\":{\"Version\":2,\"From\":[{\"Name\":\"c\",\"Entity\":\"CONSOLIDADO_TOTAL\",\"Type\":0}],\"Select\":[{\"Aggregation\":{\"Expression\":{\"Column\":{\"Expression\":{\"SourceRef\":{\"Source\":\"c\"}},\"Property\":\"Total do Município\"}},\"Function\":4},\"Name\":\"Sum(CONSOLIDADO_TOTAL.Total do Município*)\",\"NativeReferenceName\":\"Máximo de Total do Município*\"}],\"Where\":[{\"Condition\":{\"In\":{\"Expressions\":[{\"Column\":{\"Expression\":{\"SourceRef\":{\"Source\":\"c\"}},\"Property\":\"data extração\"}}],\"Values\":[[{\"Literal\":{\"Value\":\"datetime'2026-06-02T00:00:00'\"}}]]}}}],\"OrderBy\":[{\"Direction\":2,\"Expression\":{\"Aggregation\":{\"Expression\":{\"Column\":{\"Expression\":{\"SourceRef\":{\"Source\":\"c\"}},\"Property\":\"Total do Município\"}},\"Function\":4}}}]},\"Binding\":{\"Primary\":{\"Groupings\":[{\"Projections\":[0]}]},\"DataReduction\":{\"DataVolume\":3,\"Primary\":{\"Window\":{}}},\"Version\":1},\"ExecutionMetricsKind\":1}}]}","QueryId":"","ApplicationContext":{"DatasetId":"fe2d5e79-1336-4a24-8fc2-d46a7a7de9d9","Sources":[{"ReportId":"6a4ddc79-bcfd-49c2-b084-4bb4fedbda32","VisualId":"51f2d5a074ac3fb491ce"}]}}],"cancelQueries":[],"modelId":7702428}

# Exemplo de como colar:
# PAYLOAD_CAPTURADO = '''
# { "version": "1.0.0", "queries": [ { ... } ] }
# '''

# Máximo de linhas por requisição (Power BI limita a ~30.000)
TOP_N = 100

# Pasta de saída dos arquivos
OUTPUT_DIR = "/content/output"

print('✅ Configuração carregada.')
print(f'   Tabela alvo : {TABELA_ALVO}')
print(f'   Endpoint    : {API_ENDPOINT}')
print(f'   Payload     : {"capturado do DevTools" if PAYLOAD_CAPTURADO else "genérico automático"}')

✅ Configuração carregada.
   Tabela alvo : CONSOLIDADO_TOTAL
   Endpoint    : https://wabi-brazil-south-b-primary-api.analysis.windows.net/public/reports/querydata?synchronous=true
   Payload     : capturado do DevTools


In [28]:
HEADERS_EXTRAS = {
"Accept": 'application/json, text/plain, */*',
"Accept-Encoding": 'gzip, deflate, br, zstd',
"Accept-Language": 'pt-PT,pt;q=0.9',
"ActivityId": '46eea2f9-629b-0010-dd07-2ac05b24e7cf',
"Connection": 'keep-alive',
"Content-Length": '2222',
"Content-Type": 'application/json;charset=UTF-8',
"Host": 'wabi-brazil-south-b-primary-api.analysis.windows.net',
"Origin": 'https://app.powerbi.com',
"Referer": 'https://app.powerbi.com/',
"RequestId": 'f71233b8-cae0-126a-e364-043964109bba',
"Sec-Fetch-Dest": 'empty',
"Sec-Fetch-Mode": 'cors',
"Sec-Fetch-Site": 'cross-site',
"User-Agent": 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36',
"X-PowerBI-ResourceKey": '4a13952c-9b29-47cb-9601-e9b5a3ee9039',
"sec-ch-ua": '"Chromium";v="148", "Google Chrome";v="148", "Not/A)Brand";v="99"',
"sec-ch-ua-mobile": '?0',
"sec-ch-ua-platform": '"Windows"'
}

In [30]:
# ══════════════════════════════════════════════════════════════
#  PREENCHA AS VARIÁVEIS ABAIXO COM OS DADOS DO SEU DASHBOARD
# ══════════════════════════════════════════════════════════════

# URL pública do dashboard (app.powerbi.com/view?r=...)
DASHBOARD_URL = "https://app.powerbi.com/view?r=eyJrIjoiNGExMzk1MmMtOWIyOS00N2NiLTk2MDEtZTliNWEzZWU5MDM5IiwidCI6IjNhNzhiMGNkLTdjOGUtNDkyOS04M2Q1LTE5MGE2Y2MwMTM2NSJ9"

# Endpoint capturado no DevTools (URL do POST para querydata)
API_ENDPOINT = "https://wabi-brazil-south-b-primary-api.analysis.windows.net/public/reports/querydata?synchronous=true"

# Nome da tabela base (encontrado no HTML ou na análise do payload)
TABELA_ALVO = "CONSOLIDADO_TOTAL"

# Cole aqui o payload capturado do DevTools (como dict Python ou string JSON)
# Mantenha como None para usar o payload genérico automático
PAYLOAD_CAPTURADO = {"version":"1.0.0","queries":[{"Query":{"Commands":[{"SemanticQueryDataShapeCommand":{"Query":{"Version":2,"From":[{"Name":"c","Entity":"CONSOLIDADO_TOTAL","Type":0}],"Select":[{"Aggregation":{"Expression":{"Column":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"Total do Município"}},"Function":4},"Name":"Sum(CONSOLIDADO_TOTAL.Total do Município*)","NativeReferenceName":"Máximo de Total do Município*"}],"Where":[{"Condition":{"In":{"Expressions":[{"Column":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"data extração"}}],"Values":[[{"Literal":{"Value":"datetime'2026-06-02T00:00:00'"}}]]}}}],"OrderBy":[{"Direction":2,"Expression":{"Aggregation":{"Expression":{"Column":{"Expression":{"SourceRef":{"Source":"c"}},"Property":"Total do Município"}},"Function":4}}}]},"Binding":{"Primary":{"Groupings":[{"Projections":[0]}]},"DataReduction":{"DataVolume":3,"Primary":{"Window":{}}},"Version":1},"ExecutionMetricsKind":1}}]},"CacheKey":"{\"Commands\":[{\"SemanticQueryDataShapeCommand\":{\"Query\":{\"Version\":2,\"From\":[{\"Name\":\"c\",\"Entity\":\"CONSOLIDADO_TOTAL\",\"Type\":0}],\"Select\":[{\"Aggregation\":{\"Expression\":{\"Column\":{\"Expression\":{\"SourceRef\":{\"Source\":\"c\"}},\"Property\":\"Total do Município\"}},\"Function\":4},\"Name\":\"Sum(CONSOLIDADO_TOTAL.Total do Município*)\",\"NativeReferenceName\":\"Máximo de Total do Município*\"}],\"Where\":[{\"Condition\":{\"In\":{\"Expressions\":[{\"Column\":{\"Expression\":{\"SourceRef\":{\"Source\":\"c\"}},\"Property\":\"data extração\"}}],\"Values\":[[{\"Literal\":{\"Value\":\"datetime'2026-06-02T00:00:00'\"}}]]}}}],\"OrderBy\":[{\"Direction\":2,\"Expression\":{\"Aggregation\":{\"Expression\":{\"Column\":{\"Expression\":{\"SourceRef\":{\"Source\":\"c\"}},\"Property\":\"Total do Município\"}},\"Function\":4}}}]},\"Binding\":{\"Primary\":{\"Groupings\":[{\"Projections\":[0]}]},\"DataReduction\":{\"DataVolume\":3,\"Primary\":{\"Window\":{}}},\"Version\":1},\"ExecutionMetricsKind\":1}}]}","QueryId":"","ApplicationContext":{"DatasetId":"fe2d5e79-1336-4a24-8fc2-d46a7a7de9d9","Sources":[{"ReportId":"6a4ddc79-bcfd-49c2-b084-4bb4fedbda32","VisualId":"51f2d5a074ac3fb491ce"}]}}],"cancelQueries":[],"modelId":7702428}

# Máximo de linhas por requisição (Power BI limita a ~30.000)
TOP_N = 30000

# Pasta de saída dos arquivos
OUTPUT_DIR = "/content/output"

# ──────────────────────────────────────────────────────────────
# Extrai o ResourceKey real do JWT embutido na URL
# O Power BI usa o campo 'k' do JWT, não o token inteiro
# ──────────────────────────────────────────────────────────────
import base64, json as _json
from urllib.parse import urlparse, parse_qs

def extrair_resource_key_real(dashboard_url: str) -> str:
    parsed = urlparse(dashboard_url)
    qs = parse_qs(parsed.query)
    jwt = qs.get("r", [""])[0]
    if not jwt:
        return ""
    # Tenta decodificar o JWT base64 e extrair campo 'k'
    try:
        padded = jwt + "=" * (-len(jwt) % 4)
        decoded = base64.b64decode(padded).decode("utf-8")
        data = _json.loads(decoded)
        if "k" in data:
            return data["k"]
    except Exception:
        pass
    # Fallback: retorna o JWT inteiro
    return jwt

RESOURCE_KEY = extrair_resource_key_real(DASHBOARD_URL)

print('✅ Configuração carregada.')
print(f'   Tabela alvo  : {TABELA_ALVO}')
print(f'   Endpoint     : {API_ENDPOINT}')
print(f'   ResourceKey  : {RESOURCE_KEY}')
print(f'   Payload      : {"capturado do DevTools" if PAYLOAD_CAPTURADO else "genérico automático"}')

✅ Configuração carregada.
   Tabela alvo  : CONSOLIDADO_TOTAL
   Endpoint     : https://wabi-brazil-south-b-primary-api.analysis.windows.net/public/reports/querydata?synchronous=true
   ResourceKey  : 4a13952c-9b29-47cb-9601-e9b5a3ee9039
   Payload      : capturado do DevTools


---
## Célula 3 — Funções (apenas execute, não precisa editar)

In [31]:
import requests
import json
import csv
import os
import re
import copy
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse, parse_qs

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ──────────────────────────────────────────────
# UTILITÁRIOS
# ──────────────────────────────────────────────

def extrair_resource_key(url: str) -> str:
    """Extrai o ResourceKey da URL pública do Power BI."""
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    if "r" in qs:
        return qs["r"][0]
    match = re.search(r"reportId=([^&]+)", url)
    if match:
        return match.group(1)
    return ""


def montar_headers(resource_key: str) -> dict:
    base = {
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
        "Content-Type": "application/json;charset=UTF-8",
        "X-PowerBI-ResourceKey": resource_key,
        "Origin": "https://app.powerbi.com",
        "Referer": "https://app.powerbi.com/",
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
    }
    # Mescla com headers extras capturados do DevTools
    if "HEADERS_EXTRAS" in globals() and HEADERS_EXTRAS:
        base.update(HEADERS_EXTRAS)
    return base

def montar_payload_generico(tabela: str, top_n: int) -> dict:
    """Monta um payload genérico para buscar dados da tabela alvo sem filtros."""
    return {
        "version": "1.0.0",
        "queries": [
            {
                "Query": {
                    "Commands": [
                        {
                            "SemanticQueryDataShapeCommand": {
                                "Query": {
                                    "Version": 2,
                                    "From": [{"Name": "t", "Entity": tabela, "Type": 0}],
                                    "Select": [],
                                    "Top": top_n
                                },
                                "Binding": {
                                    "Primary": {"Groupings": [{"Projections": []}]},
                                    "DataReduction": {"DataVolume": 4, "Primary": {"Top": {}}},
                                    "Version": 1
                                }
                            }
                        }
                    ]
                },
                "CacheKey": f'"{tabela}_full_extract"',
                "QueryId": "",
                "ApplicationContext": {"DatasetId": "", "Sources": [{"ReportId": ""}]}
            }
        ],
        "cancelQueries": [],
        "modelId": 0
    }


def payload_sem_filtros(payload_original: dict, top_n: int) -> dict:
    """
    Recebe um payload do DevTools e remove filtros/agregações
    para retornar os dados brutos da tabela.
    """
    payload = copy.deepcopy(payload_original)
    try:
        for query in payload.get("queries", []):
            q = query["Query"]["Commands"][0]["SemanticQueryDataShapeCommand"]["Query"]
            q.pop("Where", None)
            q.pop("OrderBy", None)
            selects_limpos = []
            for sel in q.get("Select", []):
                if "Column" in sel:
                    selects_limpos.append(sel)
                elif "Aggregation" in sel:
                    inner = sel["Aggregation"].get("Expression", {})
                    if "Column" in inner:
                        selects_limpos.append({
                            "Column": inner["Column"],
                            "Name": sel.get("Name", "coluna")
                        })
            if selects_limpos:
                q["Select"] = selects_limpos
            q["Top"] = top_n
    except (KeyError, IndexError, TypeError) as e:
        print(f"  [AVISO] Não foi possível limpar todos os filtros: {e}")
    return payload


# ──────────────────────────────────────────────
# ANÁLISE DE PAYLOAD
# ──────────────────────────────────────────────

def analisar_payload(payload: dict):
    """Imprime a estrutura do payload: tabelas, colunas e filtros ativos."""
    print("═" * 60)
    print("  ANÁLISE DO PAYLOAD")
    print("═" * 60)
    queries = payload.get("queries", [])
    print(f"\n  Total de queries no payload: {len(queries)}")
    for i, q_obj in enumerate(queries):
        print(f"\n  ── Query #{i+1} ──")
        try:
            q = q_obj["Query"]["Commands"][0]["SemanticQueryDataShapeCommand"]["Query"]
            print("\n  Tabelas referenciadas (From):")
            for f in q.get("From", []):
                print(f"    alias='{f.get('Name')}' → entidade='{f.get('Entity')}' (Type={f.get('Type')})")
            select_list = q.get("Select", [])
            print(f"\n  Colunas selecionadas ({len(select_list)} itens):")
            for sel in select_list:
                nome = sel.get("Name", "sem nome")
                if "Column" in sel:
                    prop = sel["Column"].get("Property", "?")
                    src  = sel["Column"].get("Expression", {}).get("SourceRef", {}).get("Source", "?")
                    print(f"    [COLUNA]     {nome}  →  {src}.{prop}")
                elif "Measure" in sel:
                    print(f"    [MEDIDA]     {nome}  →  {sel['Measure'].get('Property', '?')}")
                elif "Aggregation" in sel:
                    fn  = sel["Aggregation"].get("Function", "?")
                    col = sel["Aggregation"].get("Expression", {}).get("Column", {}).get("Property", "?")
                    print(f"    [AGREGAÇÃO]  {nome}  →  {fn}({col})")
                else:
                    print(f"    [OUTRO]      {nome}  →  {list(sel.keys())}")
            where_list = q.get("Where", [])
            if where_list:
                print(f"\n  Filtros ativos ({len(where_list)}):")
                for w in where_list:
                    print(f"    {json.dumps(w, ensure_ascii=False)[:140]}")
            else:
                print("\n  Filtros: nenhum")
            top = q.get("Top")
            if top:
                print(f"\n  Limite de linhas (Top): {top}")
        except (KeyError, TypeError) as e:
            print(f"  [ERRO ao analisar query {i}]: {e}")


# ──────────────────────────────────────────────
# PARSE DA RESPOSTA
# ──────────────────────────────────────────────

def mapear_colunas(response_json: dict) -> dict:
    """Extrai o mapeamento índice→nome de coluna da resposta do Power BI."""
    mapeamento = {}
    try:
        ds_list = response_json["results"][0]["result"]["data"]["dsr"]["DS"]
        for ds in ds_list:
            for i, sel in enumerate(ds.get("Select", [])):
                nome = sel.get("N") or sel.get("Cn") or f"col_{i}"
                mapeamento[i] = nome
    except (KeyError, TypeError, IndexError):
        pass
    return mapeamento


def extrair_linhas(response_json: dict, mapa_colunas: dict = None) -> list:
    """
    Navega pela estrutura da resposta e extrai as linhas de dados.
    Suporta as duas estruturas mais comuns do Power BI.
    """
    linhas = []
    try:
        results = response_json.get("results", [])
        for result in results:

            # Estrutura 1: dsr → DS → PH → DM0 (visual queries)
            try:
                ds_list = result["result"]["data"]["dsr"]["DS"]
                for ds in ds_list:
                    for ph in ds.get("PH", []):
                        for item in ph.get("DM0", []):
                            valores = item.get("C", [])
                            if not valores:
                                continue
                            linha = {}
                            for i, val in enumerate(valores):
                                nome = (mapa_colunas or {}).get(i, f"col_{i}")
                                linha[nome] = val
                            linhas.append(linha)
                continue
            except (KeyError, TypeError):
                pass

            # Estrutura 2: data → results → tables → rows (REST API executeQueries)
            try:
                tabelas = result["result"]["data"]["results"][0]["tables"]
                for tabela in tabelas:
                    for row in tabela.get("rows", []):
                        row_limpa = {re.sub(r"^[^\[]+\[(.+)\]$", r"\1", k): v for k, v in row.items()}
                        linhas.append(row_limpa)
                continue
            except (KeyError, TypeError, IndexError):
                pass

    except Exception as e:
        print(f"  [ERRO] Falha ao parsear resposta: {e}")
    return linhas


# ──────────────────────────────────────────────
# EXTRAÇÃO PAGINADA
# ──────────────────────────────────────────────

def extrair_paginado(endpoint: str, headers: dict, payload_base: dict, top_n: int) -> list:
    """Faz múltiplas requisições com Top+Skip para obter todos os dados."""
    todas_linhas = []
    pagina = 0
    mapa = None

    print(f"  Iniciando extração paginada (top={top_n} por página)...")

    while True:
        payload = copy.deepcopy(payload_base)
        try:
            q = payload["queries"][0]["Query"]["Commands"][0]["SemanticQueryDataShapeCommand"]["Query"]
            q["Top"]  = top_n
            q["Skip"] = pagina * top_n
        except (KeyError, IndexError):
            pass

        print(f"  → Página {pagina+1} (offset={pagina*top_n})...", end=" ")
        resp = requests.post(endpoint, headers=headers, json=payload, timeout=60)

        if resp.status_code != 200:
            print(f"\n  [ERRO HTTP {resp.status_code}] {resp.text[:300]}")
            break

        data = resp.json()

        # Mapeia colunas apenas na primeira página
        if pagina == 0:
            mapa = mapear_colunas(data)

        linhas = extrair_linhas(data, mapa)

        if not linhas:
            print("nenhuma linha retornada. Fim da extração.")
            break

        todas_linhas.extend(linhas)
        print(f"{len(linhas)} linhas (total: {len(todas_linhas)})")

        if len(linhas) < top_n:
            break
        pagina += 1

    return todas_linhas


# ──────────────────────────────────────────────
# EXPORTAÇÃO
# ──────────────────────────────────────────────

def salvar_json(dados, nome_arquivo: str) -> str:
    caminho = os.path.join(OUTPUT_DIR, nome_arquivo)
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=2)
    print(f"  ✓ JSON salvo: {caminho}")
    return caminho


def salvar_csv(linhas: list, nome_arquivo: str) -> str:
    if not linhas:
        print("  [AVISO] Nenhuma linha para salvar.")
        return None
    caminho = os.path.join(OUTPUT_DIR, nome_arquivo)
    with open(caminho, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=list(linhas[0].keys()))
        writer.writeheader()
        writer.writerows(linhas)
    print(f"  ✓ CSV salvo: {caminho} ({len(linhas)} linhas)")
    return caminho


print('✅ Funções carregadas com sucesso.')

✅ Funções carregadas com sucesso.


---
## Célula 4 — Análise do Payload capturado

Inspeciona o payload do DevTools: mostra tabelas, colunas e filtros ativos.
Se `PAYLOAD_CAPTURADO = None`, mostra o payload genérico que será usado.

In [32]:
# ── Carrega e analisa o payload ──
if PAYLOAD_CAPTURADO:
    try:
        if isinstance(PAYLOAD_CAPTURADO, dict):
            payload_raw = PAYLOAD_CAPTURADO        # já é dict → usa direto
        else:
            payload_raw = json.loads(PAYLOAD_CAPTURADO)  # é string → faz o parse

        print("Payload capturado do DevTools — analisando...\n")
        analisar_payload(payload_raw)

        print("\n" + "═"*60)
        print("  PAYLOAD LIMPO (sem filtros, pronto para extração)")
        print("═"*60)
        PAYLOAD_FINAL = payload_sem_filtros(payload_raw, TOP_N)
        analisar_payload(PAYLOAD_FINAL)

    except json.JSONDecodeError as e:
        print(f"[ERRO] JSON inválido em PAYLOAD_CAPTURADO: {e}")
        print("Verifique se o conteúdo foi colado corretamente (incluindo as chaves externas {}).")
        PAYLOAD_FINAL = None
else:
    print("Nenhum payload capturado fornecido.")
    print(f"Será usado o payload genérico para a tabela: {TABELA_ALVO}\n")
    PAYLOAD_FINAL = montar_payload_generico(TABELA_ALVO, TOP_N)
    analisar_payload(PAYLOAD_FINAL)

Payload capturado do DevTools — analisando...

════════════════════════════════════════════════════════════
  ANÁLISE DO PAYLOAD
════════════════════════════════════════════════════════════

  Total de queries no payload: 1

  ── Query #1 ──

  Tabelas referenciadas (From):
    alias='c' → entidade='CONSOLIDADO_TOTAL' (Type=0)

  Colunas selecionadas (1 itens):
    [AGREGAÇÃO]  Sum(CONSOLIDADO_TOTAL.Total do Município*)  →  4(Total do Município)

  Filtros ativos (1):
    {"Condition": {"In": {"Expressions": [{"Column": {"Expression": {"SourceRef": {"Source": "c"}}, "Property": "data extração"}}], "Values": [[

════════════════════════════════════════════════════════════
  PAYLOAD LIMPO (sem filtros, pronto para extração)
════════════════════════════════════════════════════════════
════════════════════════════════════════════════════════════
  ANÁLISE DO PAYLOAD
════════════════════════════════════════════════════════════

  Total de queries no payload: 1

  ── Query #1 ──

  Tabelas r

---
## Célula 5 — Inspeção da API

Faz uma requisição de teste e mostra a estrutura completa da resposta.
Salva o JSON bruto em `/content/output/` para análise manual se necessário.

In [33]:
if PAYLOAD_FINAL is None:
    print("[ERRO] Payload inválido. Corrija a célula de configuração e reexecute.")
else:
    resource_key = extrair_resource_key(DASHBOARD_URL)
    if not resource_key:
        print("[AVISO] ResourceKey não encontrado na URL. Verifique DASHBOARD_URL.")
    else:
        print(f"  ResourceKey: {resource_key[:30]}...")

    HEADERS = montar_headers(resource_key)

    print(f"\n  Endpoint: {API_ENDPOINT}")
    print("\n  Enviando requisição de inspeção...")

    try:
        resp = requests.post(API_ENDPOINT, headers=HEADERS, json=PAYLOAD_FINAL, timeout=30)
        print(f"  Status HTTP: {resp.status_code}")

        if resp.status_code == 200:
            RESPOSTA_BRUTA = resp.json()
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            salvar_json(RESPOSTA_BRUTA, f"resposta_bruta_{ts}.json")

            # Mostra estrutura de chaves
            print("\n  Estrutura da resposta (chaves — 3 níveis):")
            def print_keys(obj, prefix="    ", depth=0):
                if depth > 3: return
                if isinstance(obj, dict):
                    for k, v in obj.items():
                        if isinstance(v, list):
                            print(f"{prefix}[{k}] → list({len(v)} items)")
                            if v: print_keys(v[0], prefix+"  ", depth+1)
                        else:
                            print(f"{prefix}[{k}] → {type(v).__name__}")
                            print_keys(v, prefix+"  ", depth+1)
                elif isinstance(obj, list) and obj:
                    print_keys(obj[0], prefix, depth)
            print_keys(RESPOSTA_BRUTA)

            # Tenta mapear colunas
            mapa = mapear_colunas(RESPOSTA_BRUTA)
            if mapa:
                print(f"\n  ✅ Colunas identificadas: {list(mapa.values())}")
            else:
                print("\n  [AVISO] Não foi possível mapear colunas automaticamente.")
                print("  → Abra o arquivo resposta_bruta_*.json para inspecionar a estrutura.")

            # Preview rápido das linhas
            linhas_preview = extrair_linhas(RESPOSTA_BRUTA, mapa)
            print(f"\n  Preview: {len(linhas_preview)} linhas extraídas nesta requisição.")
            if linhas_preview:
                print(f"  Exemplo da 1ª linha: {linhas_preview[0]}")
        else:
            print(f"  [ERRO] {resp.text[:500]}")
            print("\n  Verifique:")
            print("  - Se o endpoint foi copiado corretamente do DevTools")
            print("  - Se o ResourceKey está correto e o dashboard ainda está público")
            RESPOSTA_BRUTA = None

    except requests.exceptions.Timeout:
        print("[ERRO] Timeout — o servidor demorou mais de 30s para responder.")
        RESPOSTA_BRUTA = None
    except requests.exceptions.ConnectionError as e:
        print(f"[ERRO] Falha de conexão: {e}")
        RESPOSTA_BRUTA = None

  ResourceKey: eyJrIjoiNGExMzk1MmMtOWIyOS00N2...

  Endpoint: https://wabi-brazil-south-b-primary-api.analysis.windows.net/public/reports/querydata?synchronous=true

  Enviando requisição de inspeção...
  Status HTTP: 200
  ✓ JSON salvo: /content/output/resposta_bruta_20260603_194627.json

  Estrutura da resposta (chaves — 3 níveis):
    [jobIds] → list(1 items)
    [results] → list(1 items)
      [jobId] → str
      [result] → dict
        [data] → dict
          [timestamp] → str
          [rootActivityId] → str
          [metrics] → dict
          [fromCache] → bool
          [dsr] → dict

  [AVISO] Não foi possível mapear colunas automaticamente.
  → Abra o arquivo resposta_bruta_*.json para inspecionar a estrutura.

  Preview: 0 linhas extraídas nesta requisição.


---
## Célula 6 — Extração completa com paginação

Extrai todos os dados iterando com `Top` + `Skip` até esgotar os registros.

In [6]:
if PAYLOAD_FINAL is None or RESPOSTA_BRUTA is None:
    print("[ERRO] Execute as células anteriores com sucesso antes de extrair.")
else:
    print("═"*60)
    print("  EXTRAÇÃO COMPLETA")
    print("═"*60)

    DADOS = extrair_paginado(API_ENDPOINT, HEADERS, PAYLOAD_FINAL, TOP_N)

    print(f"\n  Total final: {len(DADOS)} linhas extraídas.")

    if DADOS:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        salvar_json(DADOS, f"dados_{TABELA_ALVO}_{ts}.json")
        CSV_PATH = salvar_csv(DADOS, f"dados_{TABELA_ALVO}_{ts}.csv")
        print("\n  ✅ Extração concluída!")
    else:
        print("\n  [AVISO] Nenhum dado extraído.")
        print("  → Inspecione o arquivo resposta_bruta_*.json para entender a estrutura.")
        print("  → Pode ser necessário ajustar o payload manualmente na Célula 2.")
        CSV_PATH = None

[ERRO] Execute as células anteriores com sucesso antes de extrair.


---
## Célula 7 — Preview dos dados em DataFrame

In [ ]:
if not DADOS:
    print("Nenhum dado disponível para visualizar.")
else:
    df = pd.DataFrame(DADOS)

    print(f"Shape: {df.shape[0]} linhas × {df.shape[1]} colunas")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:")
    print(df.dtypes.to_string())
    print(f"\nValores nulos por coluna:")
    print(df.isnull().sum().to_string())
    print("\n--- Primeiras 10 linhas ---")
    display(df.head(10))

---
## Célula 8 — Download do CSV no Google Colab

In [ ]:
from google.colab import files

if CSV_PATH and os.path.exists(CSV_PATH):
    print(f"Iniciando download: {CSV_PATH}")
    files.download(CSV_PATH)
else:
    print("[AVISO] Arquivo CSV não encontrado. Execute a extração primeiro (Célula 6).")